# BirdCLEF 2026 — Perch Submit v5 (DEBUG)
**Processes BOTH train_soundscapes + test_soundscapes. Diagnostic logits/sigmoid.**


In [ ]:
# [1] Install onnxruntime
import subprocess, sys, os, glob
def _fd(p):
    d=f"/kaggle/input/{p}"
    if os.path.isdir(d): return d
    for r,_,_ in os.walk("/kaggle/input"):
        if p in r: return r
    return d
WD=_fd("birdclef-perch-models")
try:
    import onnxruntime
except ImportError:
    wh=sorted(glob.glob(os.path.join(WD,"*.whl")))
    if wh:
        subprocess.check_call([sys.executable,"-m","pip","install","--no-deps",wh[0]],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
        import onnxruntime
print(f"onnxruntime {onnxruntime.__version__} OK")


In [ ]:
# [2] Imports + Paths
from pathlib import Path
import os, sys, time, re
import numpy as np, pandas as pd
import onnxruntime as ort
import librosa

INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working')
COMP=None
for c in [INPUT/'birdclef-2026',INPUT/'competitions'/'birdclef-2026']:
    if c.exists() and (c/'sample_submission.csv').exists():
        COMP=c; break
assert COMP is not None,'Competition dataset not found'
print(f'COMP: {COMP}')

SAMPLE_SUB=COMP/'sample_submission.csv'
TEST_DIR  =COMP/'test_soundscapes'
TRAIN_DIR =COMP/'train_soundscapes'
TAXONOMY  =COMP/'taxonomy.csv'
SUBM_OUT  =WORK/'submission.csv'

MODEL_DIR=None
for root,dirs,files in os.walk(INPUT):
    if 'perch_v2.onnx' in files:
        MODEL_DIR=Path(root); break
assert MODEL_DIR is not None,'Perch ONNX not found'

PERCH_ONNX  =MODEL_DIR/'perch_v2.onnx'
PERCH_LABELS=MODEL_DIR/'labels.csv'

SR,DURATION=32000,5
SEGMENT_SAMPLES=SR*DURATION
BATCH_SIZE=32
sopts=ort.SessionOptions()
sopts.graph_optimization_level=ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sopts.intra_op_num_threads=8


In [ ]:
# [3] Load + Count Test/Train rows
perch_sess=ort.InferenceSession(str(PERCH_ONNX),sopts,providers=["CPUExecutionProvider"])
perch_in=perch_sess.get_inputs()[0].name
perch_out=[o.name for o in perch_sess.get_outputs()]
print(f"Perch outputs: {perch_out}")

perch_labels_df=pd.read_csv(PERCH_LABELS)
taxonomy_df=pd.read_csv(TAXONOMY)
sample=pd.read_csv(SAMPLE_SUB)

SPECIES_COLS=[c for c in sample.columns if c!='row_id']
N_SPECIES=len(SPECIES_COLS)
species_to_idx={sp:i for i,sp in enumerate(SPECIES_COLS)}

print(f"Perch: {len(perch_labels_df)} | Taxonomy: {len(taxonomy_df)}")
print(f"Species: {N_SPECIES} | Sample rows: {len(sample)}")

# COUNT Test vs Train rows in sample_submission
test_rows =sum(1 for r in sample['row_id'] if 'Test' in str(r))
train_rows=len(sample)-test_rows
other_rows=[r for r in sample['row_id'] if 'Test' not in str(r)]
print(f"ROW BREAKDOWN: Test={test_rows}, Train/Other={train_rows}")
if other_rows:
    print(f"Non-Test row_ids (first 10): {other_rows[:10]}")
if len(sample)<=10:
    print(f"All row_ids: {sample['row_id'].tolist()}")


In [ ]:
# [4] Build Perch -> BirdCLEF mapping
tax_id_to_sci={}
tax_id_to_common={}
for _,row in taxonomy_df.iterrows():
    pid=str(row["primary_label"])
    tax_id_to_sci[pid]=str(row.get("scientific_name","")).lower().strip()
    tax_id_to_common[pid]=str(row.get("common_name","")).lower().strip()

perch_labels_list=[]
for i,row in perch_labels_df.iterrows():
    lbl=str(row.get("label",row.get("scientific_name",""))).lower().strip()
    perch_labels_list.append(lbl)

bc_to_perch={}
for bc_sp in SPECIES_COLS:
    bc_idx=species_to_idx[bc_sp]
    sci=tax_id_to_sci.get(bc_sp,bc_sp.lower().replace("_"," ")).strip()
    common=tax_id_to_common.get(bc_sp,"").strip()
    matched=False
    for pi,pl in enumerate(perch_labels_list):
        if pl==sci: bc_to_perch[bc_idx]=pi; matched=True; break
    if not matched and " " in sci:
        g=sci.split()[0]
        for pi,pl in enumerate(perch_labels_list):
            if pl.startswith(g+" "): bc_to_perch[bc_idx]=pi; matched=True; break
    if not matched and common:
        for pi,pl in enumerate(perch_labels_list):
            if pl==common or common in pl or pl in common:
                bc_to_perch[bc_idx]=pi; matched=True; break
matched=len(bc_to_perch)
print(f"Matched: {matched}/{N_SPECIES}")


In [ ]:
# [5] Inference + DIAGNOSTICS (process BOTH test + train soundscapes)
MAX_TRAIN_FILES = 200  # limit for debug (10658 train files total)

DIAG={"logit_min":[],"logit_max":[],"logit_mean":[],"sig_min":[],"sig_max":[],"sig_mean":[],"active":[]}

def load_segments(fp,seg_s=5.0,tgt_sr=32000):
    y,sr=librosa.load(fp,sr=None,mono=True)
    if sr!=tgt_sr: y=librosa.resample(y,orig_sr=sr,target_sr=tgt_sr)
    slen=int(seg_s*tgt_sr)
    if len(y)<slen: y=np.pad(y,(0,slen-len(y)))
    segs=[]
    for st in range(0,len(y)-slen+1,slen): segs.append(y[st:st+slen])
    if not segs: segs.append(y[:slen])
    return [s.astype(np.float32) for s in segs]

def predict_segments(waveforms):
    wavs=np.stack(waveforms); n=len(wavs)
    all_p=np.zeros((n,N_SPECIES),dtype=np.float32)
    for i in range(0,n,BATCH_SIZE):
        batch=wavs[i:i+BATCH_SIZE]
        outs=perch_sess.run(perch_out,{perch_in:batch})
        od=dict(zip(perch_out,outs))
        logits=od.get("label",od.get("logits"))
        if logits is None: continue
        bc_logits=np.zeros((len(batch),N_SPECIES),dtype=np.float32)
        for bc_idx,pi in bc_to_perch.items():
            if pi<logits.shape[1]: bc_logits[:,bc_idx]=logits[:,pi]
        DIAG["logit_min"].append(float(bc_logits.min()))
        DIAG["logit_max"].append(float(bc_logits.max()))
        DIAG["logit_mean"].append(float(bc_logits.mean()))
        p=1.0/(1.0+np.exp(-bc_logits))
        DIAG["sig_min"].append(float(p.min()))
        DIAG["sig_max"].append(float(p.max()))
        DIAG["sig_mean"].append(float(p.mean()))
        DIAG["active"].append(int((p>0.01).any(axis=1).sum()))
        all_p[i:i+len(batch)]=p.astype(np.float32)
    return all_p

# Process soundscapes (limit train for debug)
submission_rows={}
t0=time.time()
total_processed=0

for dir_label, soundscape_dir, max_files in [("TEST",TEST_DIR,99999),("TRAIN",TRAIN_DIR,MAX_TRAIN_FILES)]:
    all_ogg=sorted(soundscape_dir.glob('*.ogg')) if soundscape_dir.exists() else []
    ogg_files=all_ogg[:max_files]
    skipped=len(all_ogg)-len(ogg_files)
    print(f"\n{dir_label}_soundscapes: {len(all_ogg)} total, processing {len(ogg_files)} (skipped {skipped})")
    for idx,ogg_path in enumerate(ogg_files):
        try:
            segs=load_segments(str(ogg_path),seg_s=DURATION)
            if not segs: continue
            probs=predict_segments(segs)
            stem=ogg_path.stem
            for si in range(len(segs)):
                row_id=f"{stem}_{(si+1)*DURATION}"
                submission_rows[row_id]=probs[si].tolist()
            total_processed+=1
        except Exception as e:
            print(f"  [ERR] {ogg_path.name}: {e}")
        if (idx+1)%20==0:
            ela=time.time()-t0
            eta=ela/(idx+1)*len(ogg_files) if ogg_files else 0
            print(f"  [{idx+1:3d}/{len(ogg_files)}] {ela:.0f}s/~{eta:.0f}s")
    print(f"  {dir_label} done: {total_processed} files in {time.time()-t0:.0f}s")

runtime=time.time()-t0
print(f"\nTotal: {total_processed} soundscapes, {len(submission_rows)} segments in {runtime:.0f}s")
if not submission_rows:
    print("WARNING: No soundscapes processed (dry-run). Real audio injected during scoring.")

In [ ]:
# [6] Print diagnostics
print("\n"+"="*50)
print("PERCH DIAGNOSTICS")
print("="*50)
if DIAG["logit_min"]:
    print(f"LOGITS (mapped BirdCLEF):")
    print(f"  Min:    {min(DIAG['logit_min']):.2f}")
    print(f"  Max:    {max(DIAG['logit_max']):.2f}")
    print(f"  Mean:   {np.mean(DIAG['logit_mean']):.2f}")
    print(f"SIGMOID probabilities:")
    print(f"  Min:    {min(DIAG['sig_min']):.6f}")
    print(f"  Max:    {max(DIAG['sig_max']):.6f}")
    print(f"  Mean:   {np.mean(DIAG['sig_mean']):.6f}")
    print(f"  Active segments (>0.01 any species): {sum(DIAG['active'])} / {len(DIAG['active'])*BATCH_SIZE}")
    if max(DIAG['sig_max'])<0.01:
        print("\n*** WARNING: ALL probabilities <0.01! Sigmoid may be killing Perch logits. ***")
        print("*** Consider using softmax or temperature scaling instead. ***")
else:
    print("No diagnostics (dry-run, no audio processed).")


In [ ]:
# [7] Build submission
if not submission_rows:
    print("DRY-RUN: copying sample_submission.csv as-is.")
    sample.to_csv(SUBM_OUT,index=False)
else:
    rows=[]
    filled=0
    for _,sr in sample.iterrows():
        rid=sr['row_id']
        rd={'row_id':rid}
        if rid in submission_rows:
            for cls,prob in zip(SPECIES_COLS,submission_rows[rid]):
                rd[cls]=float(prob)
            filled+=1
        else:
            for cls in SPECIES_COLS: rd[cls]=0.0
        rows.append(rd)
    sub=pd.DataFrame(rows)
    sub=sub[['row_id']+SPECIES_COLS]
    for col in SPECIES_COLS: sub[col]=sub[col].astype(float)
    sub.to_csv(SUBM_OUT,index=False,float_format='%.6f')
    print(f"Submission: {len(sub)} rows ({filled} filled from predictions, {len(sub)-filled} zeros)")
    stats=[r for r in sample['row_id'] if r in submission_rows]
    print(f"Matched row_ids: {len(stats)}/{len(sample)}")
    if len(stats)<len(sample):
        missing=[r for r in sample['row_id'] if r not in submission_rows][:10]
        print(f"Missing row_ids (first 10): {missing}")


In [ ]:
# [8] Validation
sample=pd.read_csv(SAMPLE_SUB)
sub=pd.read_csv(SUBM_OUT)
errors=[]
if sub.shape!=sample.shape: errors.append(f"Shape: {sub.shape} vs {sample.shape}")
if list(sub.columns)!=list(sample.columns): errors.append("Column mismatch")
if not sub['row_id'].equals(sample['row_id']): errors.append("row_id mismatch")
proba=sub.drop(columns=['row_id'])
if proba.isna().sum().sum()!=0: errors.append(f"NaN: {proba.isna().sum().sum()}")
if not ((proba>=0.0)&(proba<=1.0)).all().all(): errors.append("Values out of [0,1]")
if errors:
    for e in errors: print(f"FAIL: {e}")
    raise RuntimeError("Validation FAILED")
else:
    print(f"VALID: {len(sub)} rows x {len(sub.columns)} cols")
    print(f"  Row IDs match: {sub['row_id'].equals(sample['row_id'])}")
    if len(sub)>0:
        print(f"  Range: [{proba.min().min():.6f}, {proba.max().max():.6f}]")
        print(f"  Mean: {proba.mean().mean():.6f}")
        active=int((proba.max(axis=0)>0.01).sum())
        print(f"  Active species (>0.01): {active}/{N_SPECIES}")


In [ ]:
# [9] Done
print("\n"+"="*50)
print("SUBMISSION READY — perch_submit_v5")
print("="*50)
print(f"Species matched: {matched}/{N_SPECIES}")
print(f"Soundscapes processed: {total_processed if 'total_processed' in dir() else 0}")
print(f"Row IDs in submission: {len(submission_rows) if 'submission_rows' in dir() else 0}")
if 'runtime' in dir(): print(f"Runtime: {runtime:.0f}s")
print(f"\nSubmit to competition!")
